## Módulo 2 - Aprendizagem Automática para Mobilidade e Qualidade do Ar

### 1. Introdução

**Contexto**

A câmara municipal pretende prever a qualidade do ar para ajustar planos de tráfego e alertas à população.

**Tarefas**
1. Fazer EDA (estatísticas, histogramas, correlações, missing values)
2. Criar pré-processamento (normalização, imputação, split treino/teste)
3. Treinar ≥ 2 algoritmos de classificação (ex.: Logistic Regression, Random Forest) para prever `air_quality_good`
4. Treinar pelo menos um algoritmo de regressão para prever `NO2`
5. Comparar métricas e justificar a escolha do modelo final
6. Guardar modelos e métricas em ficheiros (`.pkl`, `.csv`)

### 2. Dados

Trabalhamos sobre o `data/clean_air_quality.csv` (versão tratada produzida no Módulo 1), em vez do ficheiro raw original. Vantagens:

- Dados já filtrados a Lisboa+Porto (1442 linhas)
- Colunas vazias já removidas
- Permite comparação directa com a Rede Bayesiana do Módulo 1 no final

**Ponto de atenção:** a coluna `air_quality_good` foi **recalculada no Módulo 1** com a seguinte fórmula:

`air_quality_good = False  quando  (NO2 >= 30 µg/m³) AND (humidade >= 80%)`

Razão: a coluna original era sempre `True` em Lisboa+Porto, sem variação para treinar modelos. A nova fórmula introduz variabilidade (~10% de casos "má"). Como NO2 e humidade definem a target, vão ter que ser excluídas das features na classificação (caso contrário data leakage).

### 3. Configuração do Ambiente e Importação de Dados

#### 3.1. Importação de Bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Configurações de display
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 2)
sns.set_palette('husl')
plt.rcParams['figure.dpi'] = 100

In [ ]:
# ----------------------------------------------------------
# Paths — alterar aqui se mudarmos a localização dos ficheiros
# ----------------------------------------------------------
DATA_CSV    = '../data/clean_air_quality.csv'   # input (vem do Módulo 1)
METRICS_CSV = 'metrics.csv'                     # output partilhado pelos .py
MODEL_DIR   = '.'                               # onde guardar os .pkl

# Outputs gerados (CSV) para os respectivos train_*.py
CLASSIFICATION_DF = '../data/classification_data_clean.csv'
REGRESSION_DF     = '../data/regression_data_clean.csv'

#### 3.2. Carregamento do Dataset

In [ ]:
# Carregar o dataset limpo (gerado pelo Módulo 1)
df = pd.read_csv(DATA_CSV, sep=';')

print(f"Dataset: {df.shape[0]} linhas × {df.shape[1]} colunas")
print(f"Cidades : {df['city'].unique().tolist()}")
print(f"Período : meses {sorted(df['month'].unique().tolist())}")
print(f"\nPrimeiras linhas:")
df.head()

In [ ]:
# Estrutura geral (tipos das colunas, nulos, memória)
df.info()

#### 3.3. Tratamento dos dados

In [ ]:
df.describe().T

In [ ]:
# Distribuição da precipitação e da temperatura
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.histplot(df['precipitation_mm'], bins=20, kde=True)
plt.title('Distribuição da Precipitação')
plt.subplot(1, 2, 2)
sns.histplot(df['temperature_c'], bins=20, kde=True)
plt.title('Distribuição da Temperatura')
plt.tight_layout()
plt.show()

In [ ]:
# Histogramas para todas as variáveis numéricas
num_cols = df.select_dtypes(include=[np.number]).columns
df[num_cols].hist(bins=15, figsize=(15, 10), layout=(4, 4))
plt.suptitle('Distribuição das Variáveis Numéricas', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# Boxplot de PM2.5 por cidade
plt.figure(figsize=(12, 6))
sns.boxplot(x='city', y='PM2.5', data=df)
plt.title('Boxplot de PM2.5 por Cidade')
plt.xlabel('Cidade')
plt.ylabel('PM2.5')
plt.show()

In [ ]:
# Boxplots de todas as variáveis numéricas
# (cada uma na sua subplot porque as escalas são muito diferentes)
num_cols = df.select_dtypes(include=[np.number]).columns

ncols = 4
nrows = (len(num_cols) + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(15, nrows * 3))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.boxplot(y=df[col], ax=axes[i])
    axes[i].set_title(col)
    axes[i].set_ylabel('')

# esconder subplots vazios (caso o nº de variáveis não seja múltiplo de ncols)
for j in range(len(num_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Boxplots das variáveis numéricas', fontsize=14, y=1.00)
plt.tight_layout()
plt.show()

**Análise dos boxplots:**

Vários outliers visíveis em `PM2.5`, `PM10`, `NO2`, `SO2`, `CO` — mas correspondem a eventos reais de poluição (são exactamente os casos de "má qualidade" que queremos prever). Não removemos.

Os outliers de `pressure_hpa` e `precipitation_mm` são eventos meteorológicos reais (frentes de baixa pressão, chuva pontual) — também mantemos.

O boxplot de `year` é vazio (constante 2025) e o de `month` mostra um "outlier" no 10 que é só desbalanço no nº de linhas (mais Setembro que Outubro). Ambas as colunas serão removidas na fase de tratamento.

## TRATAMENTO DE DADOS (Classificação vs Regressão)

Optámos por usar:

- **Classificação** (target = `air_quality_good`):
    - Logistic Regression — classificador linear
    - Random Forest Classifier — classificador não-linear (ensemble)
    - KNN — classificador baseado em distância (3º para ir além do mínimo de 2)

- **Regressão** (target = `NO2`):
    - Linear Regression — regressor linear
    - Random Forest Regressor — regressor não-linear (ensemble)

Assim, o tratamento de dados varia por tarefa:

| Passo | Classificação | Regressão |
|---|---|---|
| Target | `air_quality_good` | `NO2` |
| Excluir features (leakage) | `NO2`, `humidity_percent` | `air_quality_good` |
| Excluir colunas não úteis | `year`, `month`, `city`, `datetime` | iguais |
| Split treino/teste | `train_test_split(stratify=y)` | `train_test_split` (sem `stratify`) |
| Métricas | accuracy, precision, recall, F1, ROC-AUC | R², MSE, MAE |

In [ ]:
# Matriz de correlação
plt.figure(figsize=(10, 8))
corr_matrix = df.select_dtypes(include=[np.number, 'bool']).corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Matriz de Correlação')
plt.show()

**Análise das correlações com cada target:**

- **`air_quality_good` (target da classificação):** as features `year`, `month`, `precipitation_mm`, `wind_direction_deg`, `pressure_hpa` e `O3` têm muito pouca correlação e provavelmente serão pouco informativas para a previsão.

- **`NO2` (target da regressão):** `pressure_hpa`, `wind_direction_deg`, `precipitation_mm`, `year` e `month` têm baixo grau de correlação e provavelmente não vão ter influência. As features mais importantes serão `CO`, `PM2.5`, `SO2`, `PM10` e `O3`.

### Pré-processamento comum (aplicado a ambas as tarefas)

Antes de separar em datasets de classificação e regressão, fazemos uma transformação que serve as duas:

**Extrair `hora` e `dia_semana` da coluna `datetime`** — em poluição urbana a hora do dia importa muito (horas de ponta = mais tráfego = mais NO2/PM2.5). Depois apagamos a coluna `datetime` original (string, sem utilidade directa).

In [ ]:
# Extrair hora e dia_semana do datetime (formato: 'dd/mm/yy HH:MM')
df['datetime']    = pd.to_datetime(df['datetime'], format='%d/%m/%y %H:%M')
df['hora']        = df['datetime'].dt.hour          # 0-23
df['dia_semana']  = df['datetime'].dt.dayofweek    # 0=segunda, 6=domingo

print(f"hora — valores: {sorted(df['hora'].unique().tolist())}")
print(f"dia_semana — valores: {sorted(df['dia_semana'].unique().tolist())}")
df[['datetime','hora','dia_semana']].head()

### Tratamento para Classificação

In [ ]:
# DataFrame para classificação:
# - target: air_quality_good
# - drop leak (NO2, humidity_percent) — definem a target
# - drop não-úteis (year=constante, month=só 2 valores, city=só 2 valores, datetime=já extraído)
LEAK_CLASSIFICATION = ['NO2', 'humidity_percent']
df_classification = df.drop(columns=LEAK_CLASSIFICATION + ['year', 'month', 'city', 'datetime'])
df_classification.info()

In [ ]:
# Verificação de nulos e duplicados (só para confirmar novamente que não há)
print(f"Nulos por coluna:\n{df.isnull().sum()}")
print(f"\nNúmero de linhas duplicadas: {df.duplicated().sum()}")

In [ ]:
# Exportar df_classification para CSV (para o train_classification.py)
df_classification.to_csv(CLASSIFICATION_DF, index=False)
print(f"DataFrame para classificação exportado para: {CLASSIFICATION_DF}")

### Tratamento para Regressão

In [ ]:
# DataFrame para regressão:
# - target: NO2 (numérica)
# - drop leak (air_quality_good) — deriva de NO2
# - drop não-úteis (year, month, city, datetime — mesmas razões da classificação)
LEAK_REGRESSION = ['air_quality_good']
df_regression = df.drop(columns=LEAK_REGRESSION + ['year', 'month', 'city', 'datetime'])
df_regression.info()

In [ ]:
# Exportar df_regression para CSV (para o train_regression.py)
df_regression.to_csv(REGRESSION_DF, index=False)
print(f"DataFrame para regressão exportado para: {REGRESSION_DF}")

### Resumo

A EDA produz dois ficheiros prontos para os scripts de treino:

| Ficheiro | Linhas | Colunas | Target | Consumido por |
|---|---|---|---|---|
| `data/classification_data_clean.csv` | 1442 | ~13 | `air_quality_good` | `train_classification.py` |
| `data/regression_data_clean.csv`    | 1442 | ~14 | `NO2`              | `train_regression.py`    |

**Decisões de pré-processamento aplicadas:**

- Filtragem de Lisboa+Porto e remoção de colunas vazias — feito no Módulo 1 (`clean_air_quality.csv`)
- Recálculo da target `air_quality_good` — feito no Módulo 1 (NO2 + humidade)
- Extracção de `hora` e `dia_semana` da coluna `datetime` — feito acima
- Remoção de `year` (constante), `month` (só 2 valores), `city` (só 2 valores), `datetime` (já extraído)
- Remoção de features que definem a target em cada tarefa (leakage)

**O que ainda não fizemos (fica para os `.py`):**

- Split treino/teste
- Normalização (StandardScaler dentro do Pipeline)
- Treino dos 5 modelos:
    - Classificação: Logistic Regression, Random Forest Classifier, KNN
    - Regressão: Linear Regression, Random Forest Regressor
- Cálculo de métricas e gravação em `metrics.csv`